In [5]:
import sqlparse

In [47]:
def build_cte_query(tables, primary_keys, partition_keys, target_key):
    def format_columns(cols, suffix=""):
        return [f"{col} AS {col}{suffix}" if suffix else col for col in cols]

    ctes = []
    master, slaves = tables[0], tables[1:]

    # CTEs
    def build_cte(alias, table, suffix=""):
        full_name = f"{table['database']}.{table['table']}"
        cols = format_columns(primary_keys)
        cols += format_columns(table["columns"], suffix)
        cols += partition_keys
        return f"{alias} AS (SELECT {', '.join(cols)} FROM {full_name})"

    ctes.append(build_cte("t0", master))
    for i, t in enumerate(slaves, 1):
        suffix = f"_{t['acronym']}" if t["acronym"] else ""
        ctes.append(build_cte(f"t{i}", t, suffix))

    # Joins and Select
    select_cols = [f"t0.{col}" for col in primary_keys + master["columns"]]
    joins = ""
    for i, t in enumerate(slaves, 1):
        joins += f"LEFT JOIN t{i} ON " + " AND ".join([f"t0.{k} = t{i}.{k}" for k in primary_keys + partition_keys]) + "\n"
        select_cols += [f"t{i}.{col}_{t['acronym']}" for col in t["columns"]]

    select_cols += [f"t0.{col}" for col in [target_key] + partition_keys]

    return (
        f"WITH {', '.join(ctes)}\n"
        f"SELECT {', '.join(select_cols)}\n"
        f"FROM t0 {joins}"
    )


In [52]:
tables = [
    {
        "database": "mydb",
        "table": "master_table",
        "acronym": "",
        "columns": ["name", "target"]
    },
    {
        "database": "mydb",
        "table": "slave1_table",
        "acronym": "s1",
        "columns": ["value1"]
    },
    {
        "database": "mydb",
        "table": "slave2_table",
        "acronym": "s2",
        "columns": ["value2"]
    }
]

target_key = "target"

primary_keys = ["pk1", "pk2"]
partition_keys = ["pt1"]

print(sqlparse.format(sql=build_cte_query(tables, primary_keys, partition_keys, target_key), reindent=True))


WITH t0 AS
  (SELECT pk1,
          pk2,
          name,
          target,
          pt1
   FROM mydb.master_table),
     t1 AS
  (SELECT pk1,
          pk2,
          value1 AS value1_s1,
          pt1
   FROM mydb.slave1_table),
     t2 AS
  (SELECT pk1,
          pk2,
          value2 AS value2_s2,
          pt1
   FROM mydb.slave2_table)
SELECT t0.pk1,
       t0.pk2,
       t0.name,
       t0.target,
       t1.value1_s1,
       t2.value2_s2,
       t0.target,
       t0.pt1
FROM t0
LEFT JOIN t1 ON t0.pk1 = t1.pk1
AND t0.pk2 = t1.pk2
AND t0.pt1 = t1.pt1
LEFT JOIN t2 ON t0.pk1 = t2.pk1
AND t0.pk2 = t2.pk2
AND t0.pt1 = t2.pt1
